In [2]:
# Core FL and ML libraries
%pip install flwr[simulation] torch torchvision transformers datasets peft
# For efficiency during baseline training
%pip install bitsandbytes accelerate


  Using cached torchvision-0.26.0-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
  Using cached flwr-1.27.0-py3-none-any.whl.metadata (14 kB)
  Using cached sqlalchemy-2.0.48-cp312-cp312-win_amd64.whl.metadata (9.8 kB)
  Using cached alembic-1.18.4-py3-none-any.whl.metadata (7.2 kB)
  Using cached cryptography-46.0.6-cp311-abi3-win_amd64.whl.metadata (5.7 kB)
  Using cached grpcio_health_checking-1.78.0-py3-none-any.whl.metadata (1.2 kB)
  Using cached iterators-0.0.2-py3-none-any.whl.metadata (2.5 kB)
  Using cached pycryptodome-3.23.0-cp37-abi3-win_amd64.whl.metadata (3.5 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-win_amd64.whl.metadata (2.4 kB)
  Using cached ray-2.51.1-cp312-cp312-win_amd64.whl.metadata (21 kB)
  Using cached tomli-2.4.1-cp312-cp312-win_amd64.whl.metadata (10 kB)
  Using cached tomli_w-1.2.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached msgpack-1.1.2-cp312-cp312-win_amd64.whl.metadata (8.4 kB)
  Using cached cffi-2.0.0-cp312-cp312-win_amd64.whl.metadata (2.6 kB)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
apache-airflow-core 3.1.3 requires packaging>=25.0, but you have packaging 24.2 which is incompatible.
googleapis-common-protos 1.63.2 requires protobuf!=3.20.0,!=3.20.1,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0.dev0,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
google-api-core 1.34.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<4.0.0dev,>=3.19.5, but you have protobuf 6.33.6 which is incompatible.
proto-plus 1.24.0 requires protobuf<6.0.0dev,>=3.19.0, but you have protobuf 6.33.6 which is incompatible.
pyopenssl 24.3.0 requires cryptography<45,>=41.0.5, but you have c


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
#base model 
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
# Note: For the baseline, we use standard LoRA (PEFT) to make training feasible
config = LoraConfig(r=8, lora_alpha=32, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM")

tokenizer_config.json: 0.00B [00:00, ?B/s]

f:\Python\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in F:\Huggingface\hub\models--TinyLlama--TinyLlama-1.1B-Chat-v1.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance,

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [4]:
import torch
import flwr as fl

# Flower client for local training/eval
class HealthClient(fl.client.NumPyClient):
    def __init__(self, model, train_loader, val_loader, device="cpu"):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device

    def get_parameters(self):
        return [val.cpu().numpy() for val in self.model.state_dict().values()]

    def set_parameters(self, parameters):
        state_dict = {
            k: torch.tensor(v).to(self.device)
            for k, v in zip(self.model.state_dict().keys(), parameters)
        }
        self.model.load_state_dict(state_dict, strict=False)

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        self.model.train()

        optimizer = torch.optim.Adam(self.model.parameters(), lr=float(config.get("lr", 1e-4)))
        loss_fn = torch.nn.CrossEntropyLoss()

        epochs = int(config.get("epochs", 1))
        for _ in range(epochs):
            for batch in self.train_loader:
                # adapt tensor unpacking to your dataloader output
                x, y = batch
                x, y = x.to(self.device), y.to(self.device)

                optimizer.zero_grad()
                outputs = self.model(x)
                loss = loss_fn(outputs.logits if hasattr(outputs, "logits") else outputs, y)
                loss.backward()
                optimizer.step()

        return self.get_parameters(), len(self.train_loader.dataset), {}

    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        self.model.eval()

        loss_fn = torch.nn.CrossEntropyLoss()
        loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for batch in self.val_loader:
                x, y = batch
                x, y = x.to(self.device), y.to(self.device)
                outputs = self.model(x)
                logits = outputs.logits if hasattr(outputs, "logits") else outputs
                loss += loss_fn(logits, y).item() * x.size(0)
                preds = logits.argmax(dim=-1)
                correct += (preds == y).sum().item()
                total += x.size(0)

        return float(loss / total), total, {"accuracy": float(correct / total)}

# Server strategy: FedAvg
strategy = fl.server.strategy.FedAvg()

# (Optional) Start Flower server
# fl.server.start_server(
#     server_address="0.0.0.0:8080",
#     config=fl.server.ServerConfig(num_rounds=3),
#     strategy=strategy,
# )

In [ ]:
# (Optional) Start Flower server
fl.server.start_server(
     server_address="0.0.0.0:8080",
     config=fl.server.ServerConfig(num_rounds=3),
     strategy=strategy,
 )

	Instead, use the `flower-superlink` CLI command to start a SuperLink as shown below:

		$ flower-superlink --insecure

	To view usage and all available options, run:

		$ flower-superlink --help

	Using `start_server()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
INFO :      Starting Flower server, config: num_rounds=3, no round_timeout
INFO :      Flower ECE: gRPC server running (3 rounds), SSL is disabled
INFO :      [INIT]
INFO :      Requesting initial parameters from one random client
